# AeroTrack — Part 1: Single-Video Detection, Tracking and Tracking-Log Export

## Objective

Run YOLO person detection and BoT-SORT tracking on **one video at a time**. This notebook creates all inputs required by Part 7:

- An annotated MP4 showing temporary BoT-SORT tracker IDs.
- A detailed frame-level tracking CSV containing one row per tracked detection and its bounding box.
- A tracker summary and per-frame statistics.
- `outputs/active_video_config.json`, which Parts 7–10 read automatically so their video and log paths stay aligned.

The default input is `data/people.mp4`. Change `VIDEO_FILENAME` in Section 1 when testing another single video.

**Run this notebook from the AeroTrack project root.**

---
### What changed in this fixed version (why IDs used to change)
1. **Low-confidence detections were thrown away before the tracker saw them.** The old run used `conf=0.25`, equal to BoT-SORT's high threshold, so the tracker's second (low-score) association pass never had anything to work with. Tiny people far below the drone flicker in confidence, so every flicker became a new ID. Now `conf=0.08` and the tracker splits high/low scores itself.
2. **Tracker memory was 1 second** (`track_buffer=30`). It is now 3 seconds (90 frames) with a looser match threshold.
3. **Higher detector resolution** (`imgsz=1280`): people are only ~25-60 px tall in a 1080p drone frame.
4. **Camera-motion compensation + world coordinates.** The drone moves more than 1000 px during the clip. A frame-to-frame homography is estimated and every detection's foot point is expressed in frame-1 coordinates (`world_x`, `world_y`, `world_h`). Part 7 uses this to reject impossible matches and to rescue weak appearance matches.
5. Single-frame blips (< `MIN_TRACKLET_FRAMES`) are removed so they do not inflate the unique count.

Needs `aerotrack_utils.py` in the project root next to the notebooks.


In [ ]:
from pathlib import Path
import json
import time

import cv2
import torch
import pandas as pd
import matplotlib.pyplot as plt

from ultralytics import YOLO
from IPython.display import display, Video

# Open this notebook from the AeroTrack project folder.
PROJECT_DIR = Path.cwd().resolve()
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Select ONE video for this experiment.
VIDEO_FILENAME = "drone_video.mov"
VIDEO_PATH = DATA_DIR / VIDEO_FILENAME
# Detector resolution: people are tiny in drone footage, so use more than 960.
IMAGE_SIZE = 1280
VIDEO_ID = f"{Path(VIDEO_FILENAME).stem}_v2_imgsz{IMAGE_SIZE}"   # new folder: baseline outputs are not overwritten

# Process the full video by default. To test a shorter segment, set a start
# time and duration here. Parts 7-10 inherit these values automatically.
VIDEO_START_SECONDS = 0.0
VIDEO_DURATION_SECONDS = None  # None = process from start time to end of video

# Keep all files for this video together, without overwriting other videos.
VIDEO_OUTPUT_DIR = OUTPUT_DIR / VIDEO_ID
VIDEO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Detector and tracker settings. This is the original pretrained baseline.
MODEL_NAME = "yolo26n.pt"
# Custom BoT-SORT settings tuned for small, flickering drone detections.
TRACKER_YAML = PROJECT_DIR / "aerotrack_botsort.yaml"
TRACKER_YAML.write_text("""tracker_type: botsort
track_high_thresh: 0.30
track_low_thresh: 0.08
new_track_thresh: 0.35
track_buffer: 90
match_thresh: 0.90
fuse_score: True
gmc_method: sparseOptFlow
proximity_thresh: 0.5
appearance_thresh: 0.25
with_reid: False
model: auto
""", encoding="utf-8")
TRACKER_CONFIG = str(TRACKER_YAML)
PERSON_CLASS_ID = 0
# Must be <= track_low_thresh so the tracker receives low-confidence detections.
CONFIDENCE_THRESHOLD = 0.08
MIN_TRACKLET_FRAMES = 5       # drop tracker blips shorter than this
CAMERA_MOTION_SCALE = 0.5     # downscale used when estimating camera motion
MAX_DETECTIONS_PER_FRAME = 300
PREVIEW_SECONDS = 5.0  # Offset from VIDEO_START_SECONDS

OUTPUT_VIDEO_PATH = VIDEO_OUTPUT_DIR / f"{VIDEO_ID}_tracked_botsort.mp4"
TRACK_LOG_PATH = VIDEO_OUTPUT_DIR / f"{VIDEO_ID}_track_log.csv"
TRACK_SUMMARY_PATH = VIDEO_OUTPUT_DIR / f"{VIDEO_ID}_tracker_summary.csv"
FRAME_SUMMARY_PATH = VIDEO_OUTPUT_DIR / f"{VIDEO_ID}_per_frame_metrics.csv"
ACTIVE_VIDEO_CONFIG_PATH = OUTPUT_DIR / "active_video_config.json"

print("Project folder:", PROJECT_DIR)
print("Video ID:", VIDEO_ID)
print("Video path:", VIDEO_PATH)
print("Output folder:", VIDEO_OUTPUT_DIR)
print("Detailed tracking log:", TRACK_LOG_PATH)


#### Check the environment and GPU

###### Use the NVIDIA GPU when PyTorch can access CUDA. If CUDA is unavailable, the notebook falls back to CPU, although processing will be considerably slower.

In [ ]:
print("PyTorch version:", torch.__version__)
print("OpenCV version:", cv2.__version__)

if torch.cuda.is_available():
    DEVICE = 0
    print("GPU available:", torch.cuda.get_device_name(0))
    print("CUDA version used by PyTorch:", torch.version.cuda)
    gpu_memory_gb = (
        torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    )
    print("GPU memory (GB):", round(gpu_memory_gb, 2))
else:
    DEVICE = "cpu"
    print("CUDA is not available. Using CPU.")

print("Selected device:", DEVICE)

#### Load and inspect the drone video

###### Check that the input file exists and can be opened. Read its resolution, frame rate, reported frame count and estimated duration.

###### The reported frame count is metadata supplied by the video decoder. The actual number of frames successfully read will be recorded during processing.

In [ ]:
# Inspect the selected video and read its metadata.
if not VIDEO_PATH.exists():
    raise FileNotFoundError(f"Input video not found: {VIDEO_PATH}")

cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    cap.release()
    raise RuntimeError(
        f"Could not open the selected video: {VIDEO_PATH}. "
        "Check that the file is a supported video and is not being used by another program."
    )

try:
    VIDEO_WIDTH = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    VIDEO_HEIGHT = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    VIDEO_FPS = float(cap.get(cv2.CAP_PROP_FPS))
    REPORTED_FRAME_COUNT = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Some video files do not report a usable frame rate. Use a fallback
    # for preview and timing calculations; the processing pass reads FPS again.
    if VIDEO_FPS <= 0:
        VIDEO_FPS = 25.0

    ESTIMATED_DURATION = (
        REPORTED_FRAME_COUNT / VIDEO_FPS
        if REPORTED_FRAME_COUNT > 0 else None
    )

    if VIDEO_WIDTH <= 0 or VIDEO_HEIGHT <= 0:
        raise RuntimeError(
            f"The video decoder returned an invalid resolution: "
            f"{VIDEO_WIDTH} x {VIDEO_HEIGHT}."
        )

    if VIDEO_START_SECONDS < 0:
        raise ValueError("VIDEO_START_SECONDS cannot be negative.")
    if VIDEO_DURATION_SECONDS is not None and VIDEO_DURATION_SECONDS <= 0:
        raise ValueError("VIDEO_DURATION_SECONDS must be positive or None.")
    if (
        ESTIMATED_DURATION is not None
        and VIDEO_START_SECONDS >= ESTIMATED_DURATION
    ):
        raise ValueError(
            f"VIDEO_START_SECONDS ({VIDEO_START_SECONDS}) is beyond the "
            f"reported video duration ({ESTIMATED_DURATION:.2f}s)."
        )

    print(f"Resolution: {VIDEO_WIDTH} x {VIDEO_HEIGHT}")
    print(f"FPS: {VIDEO_FPS:.2f}")
    print("Frames reported by decoder:", REPORTED_FRAME_COUNT)
    if ESTIMATED_DURATION is not None:
        print(f"Estimated duration: {ESTIMATED_DURATION:.1f} seconds")
    else:
        print("Estimated duration: unavailable (decoder did not report frame count)")
    print("Selected segment start:", VIDEO_START_SECONDS, "seconds")
    if VIDEO_DURATION_SECONDS is None:
        print("Selected segment duration: until end of video")
    else:
        print("Selected segment duration:", VIDEO_DURATION_SECONDS, "seconds")
finally:
    cap.release()


##### Preview the input video

###### Display one frame from the original video. This is a quick visual check that the file is being read correctly before running the detector.

In [ ]:
# PREVIEW_SECONDS is an offset from the selected segment start.
video_duration = ESTIMATED_DURATION or (VIDEO_START_SECONDS + PREVIEW_SECONDS + 1)
preview_time = min(
    VIDEO_START_SECONDS + PREVIEW_SECONDS,
    max(0.0, video_duration - (1.0 / VIDEO_FPS))
)

cap = cv2.VideoCapture(str(VIDEO_PATH))
cap.set(cv2.CAP_PROP_POS_MSEC, preview_time * 1000)
success, preview_frame = cap.read()

if not success:
    cap.set(cv2.CAP_PROP_POS_MSEC, VIDEO_START_SECONDS * 1000)
    success, preview_frame = cap.read()

cap.release()

if not success:
    raise RuntimeError("Could not read a preview frame from the selected video segment.")

plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(preview_frame, cv2.COLOR_BGR2RGB))
plt.title(f"Original video preview — {preview_time:.2f} seconds")
plt.axis("off")
plt.show()

#### Load the YOLO person detector

###### Load the pretrained YOLO26n model. For this baseline, we use the COCO `person` class (class ID 0).

###### The model weights are downloaded automatically the first time if they are not already available.

In [ ]:
model = YOLO(MODEL_NAME)

print("Loaded model:", MODEL_NAME)
print("Person class ID:", PERSON_CLASS_ID)
print("Person class name:", model.names[PERSON_CLASS_ID])

#### Test person detection on one frame

###### Run detection only (without tracking) on the preview frame. This verifies that the model detects people and draws bounding boxes before we process the full video.

###### The confidence value shown beside each box is the detector's confidence for that frame. It is not a tracking ID.

In [ ]:
preview_result = model.predict(
    source=preview_frame,
    classes=[PERSON_CLASS_ID],
    conf=0.25,
    imgsz=IMAGE_SIZE,
    device=DEVICE,
    verbose=False
)[0]

preview_detection_count = (
    len(preview_result.boxes)
    if preview_result.boxes is not None else 0
)

print("People detected in preview frame:", preview_detection_count)

preview_annotated = preview_result.plot()

plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(preview_annotated, cv2.COLOR_BGR2RGB))
plt.title("YOLO person detection preview")
plt.axis("off")
plt.show()

## Process the selected video once

This single pass produces both the annotated video and the detailed tracking log used by Part 7.

For each frame, the notebook:
1. Runs YOLO person detection and BoT-SORT tracking.
2. Records the bounding box, confidence, local tracker ID, frame number and source-video time for every tracked detection.
3. Writes the annotated frame to an MP4.
4. Records per-frame and per-track statistics.

Frame numbers in the log start at **1 within the selected segment**. `time_seconds` keeps the timestamp relative to the original source video. Parts 7–10 use the saved segment start to read the matching frames.

In [ ]:
# Reopen the video and seek to the selected segment start.
cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise RuntimeError("Could not reopen the input video.")

cap.set(cv2.CAP_PROP_POS_MSEC, VIDEO_START_SECONDS * 1000)
source_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
source_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
source_fps = cap.get(cv2.CAP_PROP_FPS)
if source_fps <= 0:
    source_fps = VIDEO_FPS if VIDEO_FPS > 0 else 25.0

max_frames = (
    int(round(VIDEO_DURATION_SECONDS * source_fps))
    if VIDEO_DURATION_SECONDS is not None
    else None
)

writer = cv2.VideoWriter(
    str(OUTPUT_VIDEO_PATH),
    cv2.VideoWriter_fourcc(*"mp4v"),
    source_fps,
    (source_width, source_height)
)
if not writer.isOpened():
    cap.release()
    raise RuntimeError(f"Could not create annotated video: {OUTPUT_VIDEO_PATH}")

TRACK_LOG_COLUMNS = [
    "video_id", "frame", "time_seconds", "track_id", "confidence",
    "x1", "y1", "x2", "y2"
]
FRAME_COLUMNS = [
    "video_id", "frame", "time_seconds", "detections",
    "tracked_detections", "tracker_ids_observed"
]

track_records = []
frame_records = []
unique_tracker_ids = set()
frames_processed = 0
frames_with_detections = 0
total_detections = 0
processing_start = time.time()

try:
    while max_frames is None or frames_processed < max_frames:
        success, frame = cap.read()
        if not success:
            break

        frames_processed += 1
        frame_time_seconds = VIDEO_START_SECONDS + frames_processed / source_fps

        result = model.track(
            source=frame,
            persist=True,
            tracker=TRACKER_CONFIG,
            classes=[PERSON_CLASS_ID],
            conf=CONFIDENCE_THRESHOLD,
            imgsz=IMAGE_SIZE,
            max_det=MAX_DETECTIONS_PER_FRAME,
            device=DEVICE,
            verbose=False
        )[0]

        detection_count = len(result.boxes) if result.boxes is not None else 0
        total_detections += detection_count
        if detection_count > 0:
            frames_with_detections += 1

        frame_ids = []
        tracked_count = 0

        if result.boxes is not None and result.boxes.id is not None:
            box_ids = result.boxes.id.int().cpu().numpy()
            box_coords = result.boxes.xyxy.cpu().numpy()
            box_scores = result.boxes.conf.cpu().numpy()
            tracked_count = len(box_ids)

            for tracker_id, coordinates, confidence in zip(
                box_ids, box_coords, box_scores
            ):
                tracker_id = int(tracker_id)
                x1, y1, x2, y2 = [float(value) for value in coordinates]
                frame_ids.append(tracker_id)
                unique_tracker_ids.add(tracker_id)

                track_records.append({
                    "video_id": VIDEO_ID,
                    "frame": frames_processed,
                    "time_seconds": float(frame_time_seconds),
                    "track_id": tracker_id,
                    "confidence": float(confidence),
                    "x1": x1,
                    "y1": y1,
                    "x2": x2,
                    "y2": y2
                })

        frame_records.append({
            "video_id": VIDEO_ID,
            "frame": frames_processed,
            "time_seconds": float(frame_time_seconds),
            "detections": int(detection_count),
            "tracked_detections": int(tracked_count),
            "tracker_ids_observed": int(len(set(frame_ids)))
        })

        writer.write(result.plot())

        if frames_processed % 100 == 0:
            elapsed = time.time() - processing_start
            print(
                f"Processed {frames_processed} frames | "
                f"detections: {total_detections} | "
                f"local tracker IDs: {len(unique_tracker_ids)} | "
                f"elapsed: {elapsed:.1f}s"
            )
finally:
    cap.release()
    writer.release()

processing_time = time.time() - processing_start

track_log_df = pd.DataFrame(track_records, columns=TRACK_LOG_COLUMNS)
frame_df = pd.DataFrame(frame_records, columns=FRAME_COLUMNS)

print("\nTracking pass completed.")
print("Frames processed:", frames_processed)
print("Frames with detections:", frames_with_detections)
print("Total person detections:", total_detections)
print("Tracked observations written:", len(track_log_df))
print("Unique local tracker IDs:", len(unique_tracker_ids))
print("Processing time (seconds):", round(processing_time, 2))
print("Annotated video:", OUTPUT_VIDEO_PATH)

if frames_processed == 0:
    raise RuntimeError("No video frames were processed. Check the selected segment.")

## Camera motion, world coordinates and blip removal

1. Remove tracker IDs seen in fewer than `MIN_TRACKLET_FRAMES` frames (not real people).
2. Estimate frame-to-frame camera motion from the static background and chain it into a world frame (frame 1).
3. Add `world_x`, `world_y` (foot point) and `world_h` (body height) to every tracked detection.

Part 7 reads these columns. Chained homographies drift slowly (a few px/s on the sample video), so world positions are trusted only over gaps of a few seconds.

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_DIR))
import numpy as np
import aerotrack_utils as U

raw_tracklets = track_log_df["track_id"].nunique()
frames_per_track = track_log_df.groupby("track_id")["frame"].nunique()
keep_ids = frames_per_track[frames_per_track >= MIN_TRACKLET_FRAMES].index
track_log_df = track_log_df[track_log_df["track_id"].isin(keep_ids)].copy()
unique_tracker_ids = set(int(i) for i in keep_ids)
print(f"Tracker IDs before blip removal: {raw_tracklets} | after: {len(keep_ids)}")

CAMERA_MOTION_PATH = VIDEO_OUTPUT_DIR / f"{VIDEO_ID}_camera_motion.npy"
CAMERA_QUALITY_PATH = VIDEO_OUTPUT_DIR / f"{VIDEO_ID}_camera_motion_quality.csv"
camera_H, camera_quality = U.estimate_camera_motion(
    VIDEO_PATH, VIDEO_START_SECONDS, max_frames=frames_processed,
    scale=CAMERA_MOTION_SCALE
)
np.save(CAMERA_MOTION_PATH, camera_H)
camera_quality.to_csv(CAMERA_QUALITY_PATH, index=False)
print(camera_quality["method"].value_counts().to_dict())
print("Median inliers per frame pair:", float(camera_quality["inliers"].median()))
weak = int((camera_quality["method"] == "identity").sum())
if weak:
    print(f"WARNING: {weak} frames fell back to identity motion (blur or featureless ground).")

track_log_df = U.add_world_coordinates(track_log_df, camera_H)
display(track_log_df.head())


## Export the detailed tracking log and experiment configuration

`<video_id>_track_log.csv` is the important hand-off to Part 7. It contains one row for every tracked person detection, including the bounding box and local tracker ID.

The same export also saves a tracker summary and frame-level metrics. Finally, `active_video_config.json` records the selected video, tracking-log path and segment start. Parts 7–10 read this file automatically.

In [ ]:
# Save the full frame-level tracking log, including required Part 7 columns.
track_log_df.to_csv(TRACK_LOG_PATH, index=False)

# Summarize each temporary BoT-SORT tracker ID.
TRACK_SUMMARY_COLUMNS = [
    "track_id", "first_frame", "last_frame", "first_seen_seconds",
    "last_seen_seconds", "frames_observed", "mean_confidence"
]

if track_log_df.empty:
    track_summary_df = pd.DataFrame(columns=TRACK_SUMMARY_COLUMNS)
else:
    track_summary_df = (
        track_log_df.groupby("track_id")
        .agg(
            first_frame=("frame", "min"),
            last_frame=("frame", "max"),
            first_seen_seconds=("time_seconds", "min"),
            last_seen_seconds=("time_seconds", "max"),
            frames_observed=("frame", "nunique"),
            mean_confidence=("confidence", "mean")
        )
        .reset_index()
        .sort_values("track_id")
    )

track_summary_df.to_csv(TRACK_SUMMARY_PATH, index=False)
frame_df.to_csv(FRAME_SUMMARY_PATH, index=False)

# The relative paths allow the project folder to be moved as one unit.
active_config = {
    "video_id": VIDEO_ID,
    "video_filename": VIDEO_FILENAME,
    "video_path": VIDEO_PATH.relative_to(PROJECT_DIR).as_posix(),
    "track_log_path": TRACK_LOG_PATH.relative_to(PROJECT_DIR).as_posix(),
    "annotated_video_path": OUTPUT_VIDEO_PATH.relative_to(PROJECT_DIR).as_posix(),
    "segment_start_seconds": float(VIDEO_START_SECONDS),
    "duration_seconds": (
        None if VIDEO_DURATION_SECONDS is None else float(VIDEO_DURATION_SECONDS)
    ),
    "source_fps": float(source_fps),
    "frames_processed": int(frames_processed),
    "frame_numbering": "1-based within the selected segment",
    "model_name": MODEL_NAME,
    "tracker_config": TRACKER_CONFIG,
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "image_size": IMAGE_SIZE,
    "camera_motion_path": CAMERA_MOTION_PATH.relative_to(PROJECT_DIR).as_posix(),
    "has_world_coordinates": True,
    "min_tracklet_frames": MIN_TRACKLET_FRAMES
}

with open(ACTIVE_VIDEO_CONFIG_PATH, "w", encoding="utf-8") as file:
    json.dump(active_config, file, indent=2)

print("Detailed tracking log:", TRACK_LOG_PATH)
print("Tracker summary:", TRACK_SUMMARY_PATH)
print("Per-frame metrics:", FRAME_SUMMARY_PATH)
print("Active video configuration:", ACTIVE_VIDEO_CONFIG_PATH)

display(track_summary_df.head(20))
display(track_log_df.head(10))

if track_log_df.empty:
    raise RuntimeError(
        "No tracked detections were exported. Part 7 cannot run until "
        "the detector/tracker produces at least one track ID."
    )

#### Review the results

###### Calculate the average processing FPS and display the key statistics.

###### Processing FPS is the number of video frames processed divided by elapsed processing time. It is not necessarily the same as the source video's playback FPS.

In [ ]:
processing_fps = (
    frames_processed / processing_time
    if processing_time > 0 else 0
)

summary = pd.DataFrame([
    {"metric": "Video ID", "value": VIDEO_ID},
    {"metric": "Source resolution", "value": f"{source_width} x {source_height}"},
    {"metric": "Source FPS", "value": round(source_fps, 2)},
    {"metric": "Frames processed", "value": frames_processed},
    {"metric": "Frames containing detections", "value": frames_with_detections},
    {"metric": "Total person detections", "value": total_detections},
    {"metric": "Tracked observations", "value": len(track_log_df)},
    {"metric": "Unique local tracker IDs", "value": len(unique_tracker_ids)},
    {"metric": "Processing time (seconds)", "value": round(processing_time, 2)},
    {"metric": "Processing speed (FPS)", "value": round(processing_fps, 2)}
])

display(summary)
print("\nPart 1 outputs are ready for Part 7.")


#### Display the annotated video

###### Open the saved output video to review whether the boxes follow people and whether tracker IDs remain stable.

In [ ]:
if OUTPUT_VIDEO_PATH.exists():
    display(Video(
        str(OUTPUT_VIDEO_PATH),
        embed=False,
        width=900
    ))
else:
    print("Annotated video was not created:", OUTPUT_VIDEO_PATH)

## Part 1 conclusion and next step

Part 1 establishes the single-video baseline for person detection and multi-object tracking. A BoT-SORT tracker ID is a temporary tracklet identifier; it does not prove that two tracklets belong to different real people.

The detailed tracking log is saved to the active video's output folder. The next notebook, **Part 7**, extracts person crops, generates OSNet appearance embeddings and compares non-overlapping tracklets from this same video.

Do not run the fine-tuning notebook as part of this workflow. The selected baseline remains `yolo26n.pt`.